In [157]:
import pandas as pd
import numpy as np

### Specifying dtype={"ZipCode": str} ensures zip codes are read as text strings, preserving leading zeros and preventing numeric formatting issues with missing or hyphenated codes.

In [158]:
df = pd.read_csv(r'C:\Users\Junayed\pandas_prac\Aug_3\messy_hotel_day3.csv', dtype = {"ZipCode": str})

In [159]:
df

,BookingID,GuestName,Email,Phone,RoomType,CheckIn,CheckOut,Nights,RatePerNight,TotalPaid,LoyaltyMember,ZipCode,SpecialRequests,Status
0,H001,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,$150.00,$450.00,Yes,02138,Late checkout,Confirmed
1,H002,Marcus Webb,marcus.webb@yahoo.com,617-555-0177,deluxe,04/02/2023,04/05/2023,3.0,"$1,250.00",$3750.00,No,10001,NaN,confirmed
2,H003,Aiyana Redcloud,aiyana.r@gmail.com,617.555.0188,Suite,04-03-2023,04-02-2023,NaN,999,$2997.00,Maybe,07030,Extra pillows,CONFIRMED
3,H004,Tobias Kruger,tobias.kruger@hotmail.com,NaN,STANDARD,2023-04-05,2023-04-08,3.0,$150.00,$450.00,y,02139,,confirmed
4,H005,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,04/01/2023,04/04/2023,3.0,$150.00,$450.00,Yes,02138,Late checkout,Confirmed
5,H006,Nia Osei,nia.osei@gmail.com,617-555-0199,Delux,04-06-2023,04-09-2023,3.0,1.2e3,$3600.00,n,60601,High floor requested,cancelled
6,H007,Ravi Patel,ravi.patel@gmail.com,617-555-0122,Suite,2023-04-06,2023-04-09,3.0,$999.00,$2997.00,0,94102,NaN,Cancelled
7,H008,Elin Sandberg,elin.sandberg@gmail.com,617-555-0155,Standard,04/07/2023,04/08/2023,1.0,$150.00,$150.00,Pending,73301,NaN,pending
8,H009,Jorge Alonso,jorge.alonso@gmail.com,617-555-0166,deluxe,04-07-2023,04-10-2023,3.0,"$1,250.00",$3750.00,Maybe,33101,NaN,Confirmed
9,H010,Ravi Patel,ravi.patel@gmail.com,617-555-0122,Suite,2023-04-06,2023-04-09,3.0,$999.00,$2997.00,0,94102,NaN,Cancelled


In [160]:
df.shape

(28, 14)

In [161]:
df.dtypes

BookingID              str
GuestName              str
Email                  str
Phone                  str
RoomType               str
CheckIn                str
CheckOut               str
Nights             float64
RatePerNight           str
TotalPaid              str
LoyaltyMember          str
ZipCode                str
SpecialRequests        str
Status                 str
dtype: object

### Room types strings include uppercase, lowercase and many unformatted style, so by using .title() we can fix that.

In [162]:
df["RoomType"] = df["RoomType"].astype(str).str.title()

In [163]:
df["RoomType"]

0     Standard
1       Deluxe
2        Suite
3     Standard
4     Standard
5        Delux
6        Suite
7     Standard
8       Deluxe
9        Suite
10    Standard
11       Suite
12      Deluxe
13    Standard
14    Standard
15       Suite
16       Delux
17    Standard
18      Deluxe
19       Suite
20    Standard
21       Suite
22      Deluxe
23    Standard
24    Standard
25       Suite
26      Deluxe
27    Standard
Name: RoomType, dtype: str

### The dates both on CheckIn and CheckOut unformatted and messy, (e.g. 04-06-2023, 2023-04-05, 04/01/2023 ). With to_datetime and format='mixed' we can fix the date style to a clear format Y-MM-DD.

- Also, to become sure if a person stayed exactly as Nights column says, I needed to calculate the stayed duration on Night_Stayed.

In [164]:
df["CheckIn"] = pd.to_datetime(df["CheckIn"], format='mixed', dayfirst=False)
df["CheckOut"] = pd.to_datetime(df["CheckOut"], format='mixed', dayfirst=False)

df["Night_Stayed"] = (df["CheckOut"] - df["CheckIn"]).dt.days

df["Night_Stayed"]

0     3.0
1     3.0
2    -1.0
3     3.0
4     3.0
5     3.0
6     3.0
7     1.0
8     3.0
9     3.0
10    1.0
11    NaN
12    3.0
13    2.0
14    1.0
15    3.0
16    3.0
17    1.0
18    3.0
19    3.0
20    1.0
21    3.0
22    3.0
23    1.0
24    1.0
25    3.0
26    3.0
27    1.0
Name: Night_Stayed, dtype: float64

In [165]:
df[["BookingID", "CheckIn", "CheckOut", "Nights", "Night_Stayed"]]

,BookingID,CheckIn,CheckOut,Nights,Night_Stayed
0,H001,2023-04-01,2023-04-04,3.0,3.0
1,H002,2023-04-02,2023-04-05,3.0,3.0
2,H003,2023-04-03,2023-04-02,NaN,-1.0
3,H004,2023-04-05,2023-04-08,3.0,3.0
4,H005,2023-04-01,2023-04-04,3.0,3.0
5,H006,2023-04-06,2023-04-09,3.0,3.0
6,H007,2023-04-06,2023-04-09,3.0,3.0
7,H008,2023-04-07,2023-04-08,1.0,1.0
8,H009,2023-04-07,2023-04-10,3.0,3.0
9,H010,2023-04-06,2023-04-09,3.0,3.0


There is few mismatch is showing after calculating the Night_Stayed.

1. On H003 the CheckOut date is earlier than the CheckIn date which is a guranteed mistake.
2. On H012 it is showing the person stayed 3 days but there is no CheckOut date, Now it is shown that the person paid 3 days total but I can't add next days on the CheckOut, I have to check it out with the hotel manager.
3. On H014 The person stayed 2 days but it is stated 5 days on Nights which is incorrect.
4. H020 has all the entry accorately, have to input the correct day instead of NaN.

In [166]:
mismatch = (df["Night_Stayed"] < 0) | (df["Nights"] != df["Night_Stayed"])

df.loc[mismatch, ["BookingID", "CheckIn", "CheckOut", "Nights", "Night_Stayed"]]

,BookingID,CheckIn,CheckOut,Nights,Night_Stayed
2,H003,2023-04-03,2023-04-02,NaN,-1.0
11,H012,2023-04-09,NaT,3.0,NaN
13,H014,2023-04-10,2023-04-12,5.0,2.0
19,H020,2023-04-13,2023-04-16,NaN,3.0


We are finding out all the collumns which has the dates but the Nights are empty.

In [167]:
fillable = df["Nights"].isna() & df["Night_Stayed"].notna() & (df["Night_Stayed"] >=0)
print(fillable)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19     True
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
dtype: bool


We can replace our old "Nights" columns with our accorate columns "Night_Stayed"

In [168]:
df.loc[fillable, "Nights"] = df.loc[fillable, "Night_Stayed"]

df[["BookingID", "Nights", "Night_Stayed"]]

,BookingID,Nights,Night_Stayed
0,H001,3.0,3.0
1,H002,3.0,3.0
2,H003,NaN,-1.0
3,H004,3.0,3.0
4,H005,3.0,3.0
5,H006,3.0,3.0
6,H007,3.0,3.0
7,H008,1.0,1.0
8,H009,3.0,3.0
9,H010,3.0,3.0


In [169]:
df[["BookingID", "RatePerNight"]]

,BookingID,RatePerNight
0,H001,$150.00
1,H002,"$1,250.00"
2,H003,999
3,H004,$150.00
4,H005,$150.00
5,H006,1.2e3
6,H007,$999.00
7,H008,$150.00
8,H009,"$1,250.00"
9,H010,$999.00


### In the RatePerNight, there is 'dollar' sign before every price and few entry don't have it. To calculate our total price we have to convert this column to numeric. There is ',' to differeciate thousand prices. We replace "dollar sign and ," and convert the column to numeric.

In [170]:
df["RatePerNight"] = pd.to_numeric(df["RatePerNight"].astype(str).str.strip().str.replace("$","", regex=False).str.replace(",","", regex=False))

In [171]:
df["RatePerNight"]

0      150.0
1     1250.0
2      999.0
3      150.0
4      150.0
5     1200.0
6      999.0
7      150.0
8     1250.0
9      999.0
10     150.0
11     999.0
12    1250.0
13     150.0
14     150.0
15     999.0
16    1250.0
17     150.0
18    1250.0
19     999.0
20     150.0
21     999.0
22    1250.0
23     150.0
24     150.0
25     999.0
26    1250.0
27     150.0
Name: RatePerNight, dtype: float64

In [172]:
df.head(5)

,BookingID,GuestName,Email,Phone,RoomType,CheckIn,CheckOut,Nights,RatePerNight,TotalPaid,LoyaltyMember,ZipCode,SpecialRequests,Status,Night_Stayed
0,H001,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,$450.00,Yes,02138,Late checkout,Confirmed,3.0
1,H002,Marcus Webb,marcus.webb@yahoo.com,617-555-0177,Deluxe,2023-04-02,2023-04-05,3.0,1250.0,$3750.00,No,10001,NaN,confirmed,3.0
2,H003,Aiyana Redcloud,aiyana.r@gmail.com,617.555.0188,Suite,2023-04-03,2023-04-02,NaN,999.0,$2997.00,Maybe,07030,Extra pillows,CONFIRMED,-1.0
3,H004,Tobias Kruger,tobias.kruger@hotmail.com,NaN,Standard,2023-04-05,2023-04-08,3.0,150.0,$450.00,y,02139,,confirmed,3.0
4,H005,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,$450.00,Yes,02138,Late checkout,Confirmed,3.0


Before calculating the TotalPrice we have to make sure the 'Nights' == 'Night_Stayed'.

In [173]:
fillnight = ((df["Night_Stayed"]).notna() & (df["Night_Stayed"] >= 0) & (df["Nights"] != df["Night_Stayed"]))

df.loc[fillnight, "Nights"] = df.loc[fillnight, "Night_Stayed"]

'TotalPaid' column also needs to be convert to numeric to calcualte the Total price.

In [174]:
df["TotalPaid"] = pd.to_numeric(df["TotalPaid"].astype(str).str.replace("$", "", regex=False).str.replace(",","", regex=False), errors = 'coerce')

df["TotalPaid_calc"] = df["RatePerNight"] * df["Nights"]
df["TotalPaid_calc"]

0      450.0
1     3750.0
2        NaN
3      450.0
4      450.0
5     3600.0
6     2997.0
7      150.0
8     3750.0
9     2997.0
10     150.0
11    2997.0
12    3750.0
13     300.0
14     150.0
15    2997.0
16    3750.0
17     150.0
18    3750.0
19    2997.0
20     150.0
21    2997.0
22    3750.0
23     150.0
24     150.0
25    2997.0
26    3750.0
27     150.0
Name: TotalPaid_calc, dtype: float64

In [175]:
df[["TotalPaid", "TotalPaid_calc"]]

,TotalPaid,TotalPaid_calc
0,450.0,450.0
1,3750.0,3750.0
2,2997.0,NaN
3,450.0,450.0
4,450.0,450.0
5,3600.0,3600.0
6,2997.0,2997.0
7,150.0,150.0
8,3750.0,3750.0
9,2997.0,2997.0


Checking any price mismatches.

In [176]:
price_mismatch = (df["TotalPaid"] - df["TotalPaid_calc"]).abs() > 0.01

df.loc[price_mismatch, ["BookingID", "RatePerNight", "Nights", "Night_Stayed", "TotalPaid", "TotalPaid_calc"]]

,BookingID,RatePerNight,Nights,Night_Stayed,TotalPaid,TotalPaid_calc


In [177]:
df.head(5)

,BookingID,GuestName,Email,Phone,RoomType,CheckIn,CheckOut,Nights,RatePerNight,TotalPaid,LoyaltyMember,ZipCode,SpecialRequests,Status,Night_Stayed,TotalPaid_calc
0,H001,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0
1,H002,Marcus Webb,marcus.webb@yahoo.com,617-555-0177,Deluxe,2023-04-02,2023-04-05,3.0,1250.0,3750.0,No,10001,NaN,confirmed,3.0,3750.0
2,H003,Aiyana Redcloud,aiyana.r@gmail.com,617.555.0188,Suite,2023-04-03,2023-04-02,NaN,999.0,2997.0,Maybe,07030,Extra pillows,CONFIRMED,-1.0,NaN
3,H004,Tobias Kruger,tobias.kruger@hotmail.com,NaN,Standard,2023-04-05,2023-04-08,3.0,150.0,450.0,y,02139,,confirmed,3.0,450.0
4,H005,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0


In [178]:
df["LoyaltyMember"].value_counts()

LoyaltyMember
Yes        6
No         5
Maybe      4
n          3
0          3
Pending    3
y          2
1          2
Name: count, dtype: int64

Converting all the loyalty tag to .title()

In [179]:
loyalty_map = {
    "yes": "Yes", "y": "Yes", "1": "Yes", 
    "no": "No", "n": "No", "0": "No",
    "maybe": "Maybe", "pending":"Maybe",
}

df["LoyaltyMember"] = df["LoyaltyMember"].astype(str).str.strip().str.lower().map(loyalty_map)
df["LoyaltyMember"].value_counts()

LoyaltyMember
No       11
Yes      10
Maybe     7
Name: count, dtype: int64

In [180]:
df.head(5)

,BookingID,GuestName,Email,Phone,RoomType,CheckIn,CheckOut,Nights,RatePerNight,TotalPaid,LoyaltyMember,ZipCode,SpecialRequests,Status,Night_Stayed,TotalPaid_calc
0,H001,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0
1,H002,Marcus Webb,marcus.webb@yahoo.com,617-555-0177,Deluxe,2023-04-02,2023-04-05,3.0,1250.0,3750.0,No,10001,NaN,confirmed,3.0,3750.0
2,H003,Aiyana Redcloud,aiyana.r@gmail.com,617.555.0188,Suite,2023-04-03,2023-04-02,NaN,999.0,2997.0,Maybe,07030,Extra pillows,CONFIRMED,-1.0,NaN
3,H004,Tobias Kruger,tobias.kruger@hotmail.com,NaN,Standard,2023-04-05,2023-04-08,3.0,150.0,450.0,Yes,02139,,confirmed,3.0,450.0
4,H005,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0


In 'SpecialRequests' there is empty cell, nan, None and this are messy. So, we need to convert everything to nan to make it clean.

In [ ]:
df["SpecialRequests"] = df["SpecialRequests"].astype(str).str.strip()
df["SpecialRequests"] = df["SpecialRequests"].replace(["","nan","None"],np.nan)
df["SpecialRequests"]

0            Late checkout
1                      NaN
2            Extra pillows
3                      NaN
4            Late checkout
5     High floor requested
6                      NaN
7                      NaN
8                      NaN
9                      NaN
10              Quiet room
11                     NaN
12         Airport shuttle
13        Nights mismatch?
14                     NaN
15                     NaN
16              High floor
17                     NaN
18                     NaN
19                Sea view
20                     NaN
21                     NaN
22            Extra towels
23                     NaN
24                     NaN
25            Balcony room
26                     NaN
27                     NaN
Name: SpecialRequests, dtype: str

In [183]:
df.head(5)

,BookingID,GuestName,Email,Phone,RoomType,CheckIn,CheckOut,Nights,RatePerNight,TotalPaid,LoyaltyMember,ZipCode,SpecialRequests,Status,Night_Stayed,TotalPaid_calc
0,H001,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0
1,H002,Marcus Webb,marcus.webb@yahoo.com,617-555-0177,Deluxe,2023-04-02,2023-04-05,3.0,1250.0,3750.0,No,10001,NaN,confirmed,3.0,3750.0
2,H003,Aiyana Redcloud,aiyana.r@gmail.com,617.555.0188,Suite,2023-04-03,2023-04-02,NaN,999.0,2997.0,Maybe,07030,Extra pillows,CONFIRMED,-1.0,NaN
3,H004,Tobias Kruger,tobias.kruger@hotmail.com,NaN,Standard,2023-04-05,2023-04-08,3.0,150.0,450.0,Yes,02139,NaN,confirmed,3.0,450.0
4,H005,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0


In [185]:
df["Status"] = df["Status"].astype(str).str.strip().str.title()

df["Status"]

0     Confirmed
1     Confirmed
2     Confirmed
3     Confirmed
4     Confirmed
5     Cancelled
6     Cancelled
7       Pending
8     Confirmed
9     Cancelled
10    Confirmed
11    Confirmed
12    Confirmed
13    Confirmed
14      Pending
15    Confirmed
16    Confirmed
17    Cancelled
18    Confirmed
19    Confirmed
20    Confirmed
21    Cancelled
22    Confirmed
23    Confirmed
24      Pending
25    Confirmed
26    Confirmed
27    Confirmed
Name: Status, dtype: str

Finding the duplicate rows.

In [186]:
dup_cols = ["GuestName", "Email", "RoomType", "CheckIn", "CheckOut"]

df[df.duplicated(subset=dup_cols, keep=False)]

,BookingID,GuestName,Email,Phone,RoomType,CheckIn,CheckOut,Nights,RatePerNight,TotalPaid,LoyaltyMember,ZipCode,SpecialRequests,Status,Night_Stayed,TotalPaid_calc
0,H001,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0
4,H005,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0
6,H007,Ravi Patel,ravi.patel@gmail.com,617-555-0122,Suite,2023-04-06,2023-04-09,3.0,999.0,2997.0,No,94102,NaN,Cancelled,3.0,2997.0
7,H008,Elin Sandberg,elin.sandberg@gmail.com,617-555-0155,Standard,2023-04-07,2023-04-08,1.0,150.0,150.0,Maybe,73301,NaN,Pending,1.0,150.0
9,H010,Ravi Patel,ravi.patel@gmail.com,617-555-0122,Suite,2023-04-06,2023-04-09,3.0,999.0,2997.0,No,94102,NaN,Cancelled,3.0,2997.0
24,H025,Elin Sandberg,elin.sandberg@gmail.com,617-555-0155,Standard,2023-04-07,2023-04-08,1.0,150.0,150.0,Maybe,73301,NaN,Pending,1.0,150.0


Removing the duplicates and reset index.

In [188]:
df = df.drop_duplicates(subset=dup_cols, keep='first').reset_index(drop=True)
df

,BookingID,GuestName,Email,Phone,RoomType,CheckIn,CheckOut,Nights,RatePerNight,TotalPaid,LoyaltyMember,ZipCode,SpecialRequests,Status,Night_Stayed,TotalPaid_calc
0,H001,Clara Jensen,clara.jensen@gmail.com,617-555-0134,Standard,2023-04-01,2023-04-04,3.0,150.0,450.0,Yes,02138,Late checkout,Confirmed,3.0,450.0
1,H002,Marcus Webb,marcus.webb@yahoo.com,617-555-0177,Deluxe,2023-04-02,2023-04-05,3.0,1250.0,3750.0,No,10001,NaN,Confirmed,3.0,3750.0
2,H003,Aiyana Redcloud,aiyana.r@gmail.com,617.555.0188,Suite,2023-04-03,2023-04-02,NaN,999.0,2997.0,Maybe,07030,Extra pillows,Confirmed,-1.0,NaN
3,H004,Tobias Kruger,tobias.kruger@hotmail.com,NaN,Standard,2023-04-05,2023-04-08,3.0,150.0,450.0,Yes,02139,NaN,Confirmed,3.0,450.0
4,H006,Nia Osei,nia.osei@gmail.com,617-555-0199,Delux,2023-04-06,2023-04-09,3.0,1200.0,3600.0,No,60601,High floor requested,Cancelled,3.0,3600.0
5,H007,Ravi Patel,ravi.patel@gmail.com,617-555-0122,Suite,2023-04-06,2023-04-09,3.0,999.0,2997.0,No,94102,NaN,Cancelled,3.0,2997.0
6,H008,Elin Sandberg,elin.sandberg@gmail.com,617-555-0155,Standard,2023-04-07,2023-04-08,1.0,150.0,150.0,Maybe,73301,NaN,Pending,1.0,150.0
7,H009,Jorge Alonso,jorge.alonso@gmail.com,617-555-0166,Deluxe,2023-04-07,2023-04-10,3.0,1250.0,3750.0,Maybe,33101,NaN,Confirmed,3.0,3750.0
8,H011,Wanjiru Kamau,wanjiru.k@gmail.com,617-555-0111,Standard,2023-04-09,2023-04-10,1.0,150.0,150.0,Yes,98101,Quiet room,Confirmed,1.0,150.0
9,H012,Felix Moreau,felix.moreau@gmail.com,617-555-0144,Suite,2023-04-09,NaT,3.0,999.0,2997.0,No,02108,NaN,Confirmed,NaN,2997.0


In [190]:
df.dtypes

BookingID                     str
GuestName                     str
Email                         str
Phone                         str
RoomType                      str
CheckIn            datetime64[us]
CheckOut           datetime64[us]
Nights                    float64
RatePerNight              float64
TotalPaid                 float64
LoyaltyMember                 str
ZipCode                       str
SpecialRequests               str
Status                        str
Night_Stayed              float64
TotalPaid_calc            float64
dtype: object

In [191]:
df.isna().sum()

BookingID           0
GuestName           0
Email               0
Phone               3
RoomType            0
CheckIn             0
CheckOut            1
Nights              1
RatePerNight        0
TotalPaid           0
LoyaltyMember       0
ZipCode             0
SpecialRequests    15
Status              0
Night_Stayed        1
TotalPaid_calc      1
dtype: int64

printing the cleaned data to a excel file.

In [192]:
df.to_excel("cleaned_hotel_data.xlsx", index=False)